# Lesson 3 - Building a RAG Agent with CrewAI

In this lesson, you will build a RAG agent with `CrewAI`. You will do that by integrating `RagTool` from `crewai_tools` with a `CrewAI` agent. `RagTool` provides a way to create and query knowledge bases from various data sources, and allows the agent to access specialized context. In this lesson, you will provide the RAG tool a pdf file containing details about PCCC coverage provided by a private health insurer. By the end of the lesson, you will build an insurer agent specialized in answering queries related to health benefits. In the next lessons, you will wrap this agent in an ACP server and make it interact with other ACP agents.

<p style="background-color:#fff6ff; padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>To Access <code>requirements.txt</code> and the <code>data</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>. For more help, please see the <em>"Appendix – Tips, Help, and Download"</em> Lesson.</p>

## 3.1. Import Libraries

In [1]:
from crewai import Crew, Task, Agent, LLM
from crewai_tools import RagTool

In [2]:
import warnings
warnings.filterwarnings('ignore')

## 3.2. Define the Agent's Large Language Model

You'll now define the large language model that you will use for your CrewAI agent. `max_tokens`: maximum number of tokens the model can generate in a single response.

**Note**: If you will define this model locally, it requires that you define the API key in a **.env** file as follows:
```
# Required
OPENAI_API_KEY=sk-...

# Optional
OPENAI_API_BASE=<custom-base-url>
OPENAI_ORGANIZATION=<your-org-id>
```

In [3]:
llm = LLM(model="openai/gpt-4", max_tokens=1024)

## 3.3. Define the RAG Tool

For the RAG tool, you can define the model provider and the embedding model in a configuration Python dictionary. You can also define the details of your vector database. If you don't specify the vector database, the RagTool will use Chroma (ChromaDB) as the default vector database in local/in-memory mode.

In [4]:
config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": "gpt-4",
        }
    },
    "embedding_model": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-ada-002"
        }
    }
}

You can then pass the `config` to the `RagTool`, and then specify the data source for which the knowledge base will be constructed. When embedding your data, the `RagTool` chunks your document into chunks and create an embedding vector for each chunk. You can specify the chunk size (`chunk_size`: number of characters) and how many characters overlap between consecutive chunks (`chunk_overlap`). You can also use the default behavior.

In [5]:
rag_tool = RagTool(config=config,  
                   chunk_size=1200,       
                   chunk_overlap=200,     
                  )
rag_tool.add("./PCCC.pdf", data_type="pdf_file")

In [6]:
rag_tool

RagTool(name='Knowledge base', description="Tool Name: Knowledge base\nTool Arguments: {'query': {'description': None, 'type': 'str'}}\nTool Description: A knowledge base that can be used to answer questions.", args_schema=<class 'abc.RagToolSchema'>, description_updated=False, cache_function=<function BaseTool.<lambda> at 0x7fbafa15b7e0>, result_as_answer=False, summarize=False, adapter=EmbedchainAdapter(embedchain_app=<embedchain.app.App object at 0x7fbaec347ed0>, summarize=False), config={'llm': {'provider': 'openai', 'config': {'model': 'gpt-4'}}, 'embedding_model': {'provider': 'openai', 'config': {'model': 'text-embedding-ada-002'}}})

## 3.4. Define the PCCC Agent

Now that you have the `rag_tool` defined, you define the CrewAI agent that can assist with PCCC coverage queries.

In [7]:
fire_safety_agent = Agent(
    role="Fire Safety Compliance Specialist", 
    goal="Answer queries about fire safety regulations and procedures accurately",
    backstory="You are a fire safety expert, knowledgeable in Vietnamese fire protection laws, inspection schedules, and safety compliance standards.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[rag_tool],  # rag_tool nên tích hợp dữ liệu về luật PCCC
    max_retry_limit=5
)

## 3.5. Define the Agent Task

Let's now test the PCCC agent. For that, you need to define the agent task and pass to it the query and the agent.

In [8]:
task1 = Task(
    description='Mức phạt tiền với các hành vi vi phạm gây cháy nổ là bao nhiêu?',
    expected_output="Một câu trả lời đầy đủ, chính xác dựa trên quy định hiện hành về kiểm tra PCCC định kỳ.",
    agent=fire_safety_agent
)

## 3.6. Run the PCCC Agent

To run the agent, you need to pass the agent and the task to a Crew object that you can run using the `kickoff` method.

In [9]:
crew = Crew(agents=[fire_safety_agent], tasks=[task1], verbose=True)
task_output = crew.kickoff()
print(task_output) 

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 91667db1-38c9-4ab9-99af-405e522439b9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 931d47ff-5f70-48d5-9dd3-6195de1235c2
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 931d47ff-5f70-48d5-9dd3-6195de1235c2
       Status: Executing Task...
    └── 🤖 Agent: Fire Safety Compliance Specialist
            Status: In Progress

# Agent: Fire Safety Compliance Specialist
## Task: Mức phạt tiền với các hành vi vi phạm gây cháy nổ là bao nhiêu?


🤖 Agent: Fire Safety Compliance Specialist
    Status: In Progress
└── 🧠 Thinking...

...

🚀 Crew: crew
└── 📋 Task: 931d47ff-5f70-48d5-9dd3-6195de1235c2
       Status: Executing Task...
    └── 🤖 Agent: Fire Safety Compliance Specialist
            Status: ✅ Completed

🚀 Crew: crew
└── 📋 Task: 931d47ff-5f70-48d5-9dd3-6195de1235c2
       Assigned to: Fire Safety Compliance Specialist
       Status: ✅ Completed
    └── 🤖 Agent: Fire Safety Compliance Specialist
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 931d47ff-5f70-48d5-9dd3-6195de1235c2                                                                     │
│  Agent: Fire Safety Compliance Specialist                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 91667db1-38c9-4ab9-99af-405e522439b9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Mức phạt tiền cho các hành vi vi phạm gây cháy, nổ tùy thuộc vào mức độ nghiêm trọng và tính chất cụ thể của lỗi vi phạm. Ví dụ, vi phạm trong việc sử dụng nguồn nhiệt có thể bị phạt tối đa 25.000.000 đồng, trong khi vi phạm liên quan đến trang thiết bị, lắp đặt, sử dụng, bảo dưỡng của phương tiện PCCC và thiết bị có thể bị phạt tối đa 50.000.000 đồng. Cần lưu ý rằng bất kỳ hành vi vi phạm nào liên quan đến việc thành lập và duy trì một đội PCCC và cứu nạn cơ sở có thể bị phạt tối đa 30.000.000 đồng. Hành vi vi phạm liên quan đến quy trình thoát hiểm có thể bị phạt tối đa 50.000.000 đồng, trong khi việc không kiểm tra về PCCC định kỳ có thể bị phạt tối đa 20.000.000 đồng. Các hoạt động không trực tiếp gây ra cháy nổ cũng bị phạt tiền nhưng với mức độ khoan hồng hơn.


## 3.7. Resources

- [CrewAI Agents](https://docs.crewai.com/concepts/agents)
- [CrewAI Tasks](https://docs.crewai.com/concepts/tasks)
- [CrewAI RagTool](https://docs.crewai.com/tools/ai-ml/ragtool)
- [Short course on Multi Agents with CrewAI](https://www.deeplearning.ai/short-courses/multi-ai-agent-systems-with-crewai/)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

</div>